## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending email</h2>
            <span style="color:#ff7800;">This lab can send email via <strong>Gmail SMTP</strong> (good if you only have a Gmail address) or <strong>SendGrid</strong>.<br/>
            The next section explains setup. Alternatives: <a href="https://edwarddonner.com/faq">FAQ Q29</a> and community Resend examples.
            </span>
        </td>
    </tr>
</table>

## Setting up email (Gmail SMTP or SendGrid)

### Option A — Gmail only (recommended if you use `@gmail.com`)

1. Turn on **2-Step Verification** for your Google account.
2. Create an **App password**: [Google App passwords](https://myaccount.google.com/apppasswords) (choose “Mail” / “Other”).
3. Add to your `.env` file:

`GMAIL_ADDRESS=you@gmail.com`  
`GMAIL_APP_PASSWORD=xxxx` (paste the 16-character password; spaces are OK)

Optional — send tests to another inbox: `GMAIL_TO_EMAIL=other@example.com` (defaults to `GMAIL_ADDRESS`).

The notebook sends through **smtp.gmail.com** so `From:` matches Gmail and delivery works.

### Option B — SendGrid

Visit https://sendgrid.com/ — create an API key and add `SENDGRID_API_KEY=xxxx`. For reliable delivery to Gmail, use **Authenticate Your Domain** and set `SENDGRID_FROM_EMAIL` / `SENDGRID_TO_EMAIL` to addresses on that domain (not `@gmail.com` as From).

__Other options:__ [FAQ Q29](https://edwarddonner.com/faq), `community_contributions/2_lab2_with_resend_email`, or skip email.

### Choosing a provider

If both are configured, Gmail is used unless you set `EMAIL_PROVIDER=sendgrid`.

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, set_tracing_disabled
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict, Literal
import sendgrid
import os
import smtplib
from email.mime.text import MIMEText
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio


In [2]:
load_dotenv(override=True)

if os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
    os.environ["OPENAI_BASE_URL"] = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
    # Traces go to api.openai.com; OpenRouter keys are not valid there (401).
    set_tracing_disabled(True)


def _email_provider() -> Literal["gmail", "sendgrid"]:
    """Prefer Gmail SMTP when configured; otherwise SendGrid. Set EMAIL_PROVIDER to force one."""
    forced = (os.getenv("EMAIL_PROVIDER") or "").strip().lower()
    if forced == "gmail":
        if not (os.getenv("GMAIL_APP_PASSWORD") and os.getenv("GMAIL_ADDRESS")):
            raise ValueError("EMAIL_PROVIDER=gmail requires GMAIL_ADDRESS and GMAIL_APP_PASSWORD in .env")
        return "gmail"
    if forced == "sendgrid":
        if not os.getenv("SENDGRID_API_KEY"):
            raise ValueError("EMAIL_PROVIDER=sendgrid requires SENDGRID_API_KEY in .env")
        if not os.getenv("SENDGRID_FROM_EMAIL") or not os.getenv("SENDGRID_TO_EMAIL"):
            raise ValueError("EMAIL_PROVIDER=sendgrid requires SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL in .env")
        return "sendgrid"
    if os.getenv("GMAIL_APP_PASSWORD") and os.getenv("GMAIL_ADDRESS"):
        return "gmail"
    if os.getenv("SENDGRID_API_KEY"):
        if not os.getenv("SENDGRID_FROM_EMAIL") or not os.getenv("SENDGRID_TO_EMAIL"):
            raise ValueError(
                "SendGrid needs SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL — or add GMAIL_ADDRESS + GMAIL_APP_PASSWORD to use Gmail SMTP instead."
            )
        return "sendgrid"
    raise ValueError(
        "Set up email in .env: GMAIL_ADDRESS + GMAIL_APP_PASSWORD (Gmail), "
        "or SENDGRID_API_KEY + SENDGRID_FROM_EMAIL + SENDGRID_TO_EMAIL (SendGrid)."
    )


def send_lab_email(subject: str, body: str, *, subtype: Literal["plain", "html"] = "plain") -> None:
    """Send one message via Gmail or SendGrid (see _email_provider)."""
    prov = _email_provider()
    if prov == "gmail":
        user = os.environ["GMAIL_ADDRESS"].strip()
        pwd = os.environ["GMAIL_APP_PASSWORD"].replace(" ", "")
        to_addr = (os.getenv("GMAIL_TO_EMAIL") or user).strip()
        msg = MIMEText(body, subtype, "utf-8")
        msg["Subject"] = subject
        msg["From"] = user
        msg["To"] = to_addr
        with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
            smtp.starttls()
            smtp.login(user, pwd)
            smtp.sendmail(user, [to_addr], msg.as_string())
        return
    from_addr = os.environ.get("SENDGRID_FROM_EMAIL")
    to_addr = os.environ.get("SENDGRID_TO_EMAIL")
    if not from_addr or not to_addr:
        raise ValueError("SendGrid needs SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL in .env")
    content_type = "text/html" if subtype == "html" else "text/plain"
    sg = sendgrid.SendGridAPIClient(api_key=os.environ["SENDGRID_API_KEY"])
    mail = Mail(Email(from_addr), To(to_addr), subject, Content(content_type, body)).get()
    response = sg.client.mail.send.post(request_body=mail)
    if response.status_code >= 400:
        raise RuntimeError(f"SendGrid HTTP {response.status_code}: {response.body}")


In [3]:
# Let's just check emails are working for you

def send_test_email():
    send_lab_email("Test email", "This is an important test email", subtype="plain")
    print("OK — check your inbox (provider:", _email_provider() + ")")


send_test_email()

OK — check your inbox (provider: gmail)


### Did you receive the test email

You should see `OK — check your inbox`. With **SendGrid**, HTTP `202` means “accepted”; with **Gmail SMTP**, there is no status code—if no exception, the send was handed off to Google.

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

- **Gmail:** Wrong app password, 2-Step Verification off, or typo in `GMAIL_ADDRESS`.
- **SendGrid:** API key or `SENDGRID_FROM_EMAIL` / `SENDGRID_TO_EMAIL` / sender verification.

Or use "Resend Email" in `community_contributions/2_lab2_with_resend_email`, Pushover, or a flat file.

## Step 1: Agent workflow

In [4]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [5]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="openai/gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="openai/gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="openai/gpt-4o-mini"
)

In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC2 Compliance Process with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. As organizations increasingly prioritize data security and compliance, ensuring SOC2 compliance has become a critical focus for many businesses, including yours.

At ComplAI, we understand that navigating the complexities of compliance can be both time-consuming and resource-intensive. Our AI-powered SaaS tool is designed to simplify this process, allowing you to efficiently manage compliance requirements, streamline audits, and ultimately save valuable time and resources.

Here’s how ComplAI can benefit your organization:

- **Automated Evidence Collection**: Our solution automates the evidence-gathering process, ensuring you have everything needed for audits at your fingertips.
- **Real-time Compliance Monitoring**: Stay ahead of potential compliance issues with real-time monitoring and alerts tailored to your specific environment.
- **Expert Guidance**: Benefi

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************c8d6. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  
[Company Website]  

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************c8d6. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************c8d6. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************c8d6. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error"

In [7]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am reaching out on behalf of ComplAI, a leading provider of AI-driven SaaS solutions designed specifically to simplify the SOC 2 compliance process.

As you may know, achieving and maintaining SOC 2 compliance can be a daunting task, often involving extensive documentation and ongoing risk assessments. Our platform automates these processes, allowing your team to focus on core business initiatives while ensuring that compliance requirements are met efficiently and effectively.

Here are a few key benefits of using ComplAI:

1. **Automated Documentation**: Our tool streamlines the documentation process, saving you valuable time and resources.
2. **Real-Time Risk Assessment**: Identify and mitigate potential risks proactively with our advanced AI-driven analytics.
3. **Seamless Audit Preparation**: Simplify your audit process with easy access

In [8]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [9]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: Ready to Make Audits Less of a Horror Show? 🎭

Hey [First Name],

Ever felt like preparing for an audit is akin to training for a marathon while juggling flaming swords? (I mean, who doesn’t love a good thrill, right?) 

At ComplAI, we believe you shouldn’t need a cape to tackle SOC2 compliance. Our AI-powered tool takes the grunt work out of the process—think of us as your trusty sidekick, minus the spandex.

Imagine breezing through audits with a smile instead of sweating bullets! Our software simplifies the compliance journey, automates documentation, and gives you the peace of mind that your security measures are not only in place but also up to date. 

Let’s turn that audit anxiety into audit excitement! (Okay, maybe not excitement…but at least manageable stress?) I’d love to show you how we can transform your compliance game. Do you have 15 minutes this week for a quick chat?

Looking forward to saving you from those audit nightmares!

Best,  
[Your Nam

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [10]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="openai/gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="openai/gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="openai/gpt-4o-mini",
)

In [11]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model='openai/gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [12]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    send_lab_email("Sales email", body, subtype="plain")
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [13]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115de3240>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [14]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1144680e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [15]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115e32340>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115e32840>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

## And now it's time for our Sales Manager - our planning agent

In [16]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [17]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [18]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    send_lab_email(subject, html_body, subtype="html")
    return {"status": "success"}

In [19]:
tools = [subject_tool, html_tool, send_html_email]

In [20]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x116020680>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115e1e840>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Function

In [21]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")


### Now we have 3 tools and 1 handoff

In [22]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115e32340>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115e32840>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write a 

In [23]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.